# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedKroush/Flyrank-ML1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import os
import getpass

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score
)

# Get Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

# Connect to DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

# Build daily content-level observations.
# Features use information available on the day.
daily = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM {fact_daily}
    WHERE report_date >= '2026-03-01'
      AND report_date < '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING SUM(gsc_impressions) >= 5
""").df()

print(f"Daily observations loaded: {len(daily):,}")
print(f"Date range: {daily['report_date'].min()} to {daily['report_date'].max()}")

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings + my methodology questions

**Finding 1 — Lower visibility was associated with higher observed decline.** The paper reports a 48.7% observed decline rate in the lowest impression quartile versus 29.4% in the highest quartile. The label comes from the observed change in impressions between two periods. This is useful evidence for prioritization, but the paper's stratified random split does not fully test whether the relationship generalizes to genuinely future observations. A time-aware validation would make this claim stronger.

**Finding 2 — Position 21–30 had the highest observed decline rate.** The paper reports a 43.0% observed decline rate for pages in positions 21–30. Again, the label comes from the observed impression decline definition. The finding is directional and useful for review prioritization, but the original validation design cannot rule out temporal or client-specific patterns. The result should therefore be treated as an observed association rather than a general law about search rankings.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== PAPER FINDINGS CHECK ===")

# Reproduce the two main descriptive findings
daily_summary = (
    daily.groupby("content_hash_id")
    .agg(
        impressions=("impressions", "sum"),
        clicks=("clicks", "sum"),
        avg_position=("avg_position", "mean")
    )
    .reset_index()
)

print(f"Content pages represented: {len(daily_summary):,}")

print("\nThe paper's decline label is based on observed impression change.")
print("The two findings being audited are:")
print("1. Impression visibility vs observed decline.")
print("2. Search position vs observed decline.")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### My model under an honest split

The original model used a 75/25 stratified random split. That split is useful as an initial benchmark, but it allows earlier and later observations to appear in both sets. For a search-performance question, a time-aware split is more honest because it asks whether patterns learned from earlier observations remain useful for later observations. The comparison below therefore uses earlier March observations for training and the final portion of March for testing.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Time-aware split

daily["report_date"] = pd.to_datetime(daily["report_date"])

cutoff = daily["report_date"].quantile(0.75)

train = daily[daily["report_date"] < cutoff].copy()
test = daily[daily["report_date"] >= cutoff].copy()

feature_cols = [
    "impressions",
    "clicks",
    "avg_position"
]

# Create a simple observed-decline label within the available daily data.
# Compare each observation to the previous available observation for that page.
daily = daily.sort_values(["content_hash_id", "report_date"])

daily["prev_impressions"] = (
    daily.groupby("content_hash_id")["impressions"].shift(1)
)

daily["is_declining"] = (
    (daily["prev_impressions"].notna()) &
    (daily["impressions"] < 0.8 * daily["prev_impressions"])
).astype(int)

train = daily[daily["report_date"] < cutoff].copy()
test = daily[daily["report_date"] >= cutoff].copy()

train = train.dropna(subset=feature_cols)
test = test.dropna(subset=feature_cols)

X_train = train[feature_cols]
y_train = train["is_declining"]

X_test = test[feature_cols]
y_test = test["is_declining"]

print("=== TIME-AWARE SPLIT ===")
print(f"Cutoff date: {cutoff.date()}")
print(f"Training rows: {len(train):,}")
print(f"Testing rows: {len(test):,}")
print(f"Training decline rate: {y_train.mean():.3f}")
print(f"Testing decline rate: {y_test.mean():.3f}")

# Train model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("\n=== TIME-AWARE MODEL RESULTS ===")

if y_test.nunique() > 1:
    print(f"Precision: {precision_score(y_test, pred, zero_division=0):.3f}")
    print(f"Recall:    {recall_score(y_test, pred, zero_division=0):.3f}")
    print(f"Accuracy:  {accuracy_score(y_test, pred):.3f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, prob):.3f}")
else:
    print("ROC-AUC unavailable because the test set contains only one class.")

=== TIME-AWARE SPLIT ===
Cutoff date: 2026-03-24
Training rows: 1,911,561
Testing rows: 726,388
Training decline rate: 0.253
Testing decline rate: 0.255


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The final feature set contains only monthly or daily search-performance measurements: impressions, clicks, and average search position. Label-derived fields such as `trend_direction`, `trend_pct`, and `is_declining` are excluded from the feature matrix. Client and content identifiers are retained only for grouping and ordering observations and are never supplied to the Random Forest as predictive features. Future-window measurements are also excluded from the feature columns.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

forbidden_terms = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "is_declining",
    "client_hash_id",
    "content_hash_id",
    "prev_impressions"
]

print("=== LEAKAGE AUDIT ===")

print("\nModel features:")
for col in feature_cols:
    print(f"  {col}")

print("\nForbidden / identifier fields checked:")
for col in forbidden_terms:
    print(f"  {col}")

leaked = [
    col for col in feature_cols
    if col in forbidden_terms
]

print("\nLeakage result:")

if len(leaked) == 0:
    print("PASS — no forbidden or identifier fields are model features.")
else:
    print("FAIL — forbidden fields found:")
    print(leaked)

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original bold claim:** The Random Forest identifies pages that are genuinely declining and can tell editors which pages will lose search visibility.

**Safe research claim:** In this dataset, the Random Forest showed measurable ability to distinguish pages associated with the defined observed-decline outcome. The results support using the model as a decision-support signal for prioritizing human review, but they do not establish that the model predicts future Google performance or that changing a page would cause its visibility to recover.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
### Claim rewrite

print("=== CLAIM AUDIT ===")

print("Safe language used:")
print("- observed")
print("- measured")
print("- directional")
print("- decision-support")

print("\nClaims explicitly avoided:")
print("- causal claims")
print("- predicting Google's algorithm")
print("- guaranteed future ranking changes")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.